# Batch regenerate Landsat lake tables

This notebook runs the table regeneration pipeline for all configured lakes, or for a selected subset. It assumes Landsat C2L1 and C2L2 products are already downloaded on the external drive and organized as `<Lake>_L1`, `<Lake>_L2_acolite`, and `<Lake>_L2_usgs` folders.

Use `TABLES = "l2"` to refresh only `data/landsat_l2_tables` after changing ML models. Use `TABLES = "both"` when you want to regenerate L1 and L2 tables together.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=None) -> Path:
    start = start or Path.cwd()
    for path in [start, *start.parents]:
        if (path / "src" / "lswt_cloud_masking").exists():
            return path
    raise RuntimeError("Run this notebook from inside the LSWT-ML-thin-cloud-masking repo.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))

from lswt_cloud_masking.lake_tables import generate_all_lake_tables, load_generation_config


def repo_path(value) -> Path:
    path = Path(value).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path


REPO_ROOT

WindowsPath('C:/Users/airani/Documents/Trishna/scripts/LSWT-ML-thin-cloud-masking')

## Settings

Change the external-drive path before running. Leave `ONLY_LAKES = None` to process every lake in the config, or set a list such as `["geneva", "mendota"]`. Set `LIMIT_SCENES = 1` for a fast smoke test. For the current seed-142 model rerun, keep `TABLES = "l2"`.

In [2]:
CONFIG_PATH = REPO_ROOT / "configs" / "lake_tables.example.json"
config = load_generation_config(CONFIG_PATH)

LANDSAT_C2_ROOT = Path(config.get("landsat_c2_root", "/Volumes/YOUR_EXTERNAL_DRIVE/Trishna/Landsat_processing/Landsat_C2"))
if not LANDSAT_C2_ROOT.exists():
    windows_root = Path(r"D:\Trishna\Landsat_processing\Landsat_C2")
    if windows_root.exists():
        LANDSAT_C2_ROOT = windows_root

TABLES = "l2"
ONLY_LAKES = None
LIMIT_SCENES = None

print(f"LANDSAT_C2_ROOT = {LANDSAT_C2_ROOT}")
print(f"TABLES = {TABLES}")

pd.DataFrame(config["lakes"])[["output_key", "lake_key", "folder_name", "numpix_xy"]]

LANDSAT_C2_ROOT = D:\Trishna\Landsat_processing\Landsat_C2
TABLES = l2


,output_key,lake_key,folder_name,numpix_xy
0,geneva,geneva,Geneva,5
1,aegeri,ageri,Aegeri,5
2,bianco,bianco,Bianco,5
3,greifensee,greifensee,Greifensee,5
4,mendota,mendota,Mendota,5
5,venice,venice,Venice,5


## Run batch processing

The tuned DT, RF, and XGBoost models are loaded once from `models/general` and reused for every lake when `TABLES` includes L2.

In [ ]:
reports = generate_all_lake_tables(
    landsat_root=LANDSAT_C2_ROOT,
    lake_geojson=repo_path(config["lake_geojson"]),
    model_dir=repo_path(config["models_dir"]),
    output_l1_dir=repo_path(config["output_l1_dir"]),
    output_l2_dir=repo_path(config["output_l2_dir"]),
    lakes=config["lakes"],
    only_lakes=ONLY_LAKES,
    mask_cloud_classes=tuple(config.get("mask_cloud_classes", [1])),
    include_cirrus_as_cloud=bool(config.get("include_cirrus_as_cloud", False)),
    include_lake_metadata_in_l1=bool(config.get("include_lake_metadata_in_l1", False)),
    limit_scenes=LIMIT_SCENES,
    continue_on_error=bool(config.get("continue_on_error", True)),
    tables=TABLES,
)

summary = pd.DataFrame(reports)
summary[["lake", "l1_rows", "l2_rows", "l1_csv", "l2_csv", "report_json"]]

C:\Users\airani\.conda\envs\landsat-cloud-review\Lib\pickle.py:1718: UserWarning: [09:57:50] WARNING: D:\bld\xgboost-split_1780806776065\work\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


## Inspect failures

When `continue_on_error` is enabled, failed scenes are listed in the per-lake report JSON and in the columns below.

In [ ]:
failure_rows = []
for report in reports:
    for stage in ["l1_failures", "l2_failures"]:
        for failure in report.get(stage, []):
            failure_rows.append({"lake": report["lake"], "stage": stage, **failure})

pd.DataFrame(failure_rows)